Vamos a tomar el  corpus_portal_completo.json + el corpus de PDFs y construir un pipeline de normalización semántica, que detecte títulos, subtítulos, listas, tablas, requisitos, pasos de procedimientos, teléfonos, correos, fechas y montos, y los convierta en fragmentos estructurados. Eso hará que los 5.000–10.000 ejemplos que generemos después sean muchísimo mejores que simplemente cortar el texto cada 2.500 caracteres .

Este pipeline será la capa intermedia entre tus documentos institucionales y el dataset de fine-tuning.

La idea será transformar esto:

PDF / Portal
     ↓
texto bruto
     ↓
normalización
     ↓
detección de estructura
     ↓
fragmentos semánticos
     ↓
metadata
     ↓
corpus_semantico.jsonl
     ↓
generación QA

No vamos a cortar simplemente cada 2500 caracteres. Un fragmento puede representar, por ejemplo:

un programa académico;
un conjunto de requisitos;
los pasos de una inscripción;
los datos de contacto de una dependencia;
una tabla de aranceles;
una definición institucional;
un conjunto de objetivos;
información de un diplomado.

Además, conservaremos fuente, URL, documento, categoría y posición, para poder rastrear posteriormente cada respuesta.

1. Estructura del proyecto

Partiendo de lo que ya hemos desarrollado:

UniversidadLLM/
│
├── documentos/
│   ├── admisiones/
│   ├── reglamentos/
│   ├── programas/
│   ├── postgrado/
│   ├── investigacion/
│   ├── servicios/
│   └── calendario/
│
├── portal/
│   ├── raw/
│   ├── clean/
│   ├── json/
│   └── corpus_portal_completo.json
│
├── texto_extraido/
├── texto_limpio/
│
├── semantico/
│
└── dataset/

Crearemos:

semantico/
├── fragmentos_semanticos.jsonl
├── fragmentos_semanticos.json
├── estadisticas.json
└── revision_manual.csv

In [1]:
# 2. Instalar dependencias
!pip install pymupdf tqdm pandas unidecode

# No vamos a introducir todavía un LLM en esta etapa.
# Primero haremos una normalización determinística, que es más reproducible y barata.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 9.8 MB/s eta 0:00:00


In [2]:
# 3. Montar el driver
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Configuración

In [3]:
import os
import re
import json
import math
import unicodedata
from collections import Counter
from datetime import datetime

import pandas as pd

from tqdm.auto import tqdm
from unidecode import unidecode

In [4]:
BASE_DIR = "/content/drive/MyDrive/UniversidadLLM"

PORTAL_CORPUS = os.path.join(
    BASE_DIR,
    "portal",
    "corpus_portal_completo.json"
)

SEMANTIC_DIR = os.path.join(
    BASE_DIR,
    "semantico"
)

os.makedirs(
    SEMANTIC_DIR,
    exist_ok=True
)

print(PORTAL_CORPUS)

/content/drive/MyDrive/UniversidadLLM/portal/corpus_portal_completo.json


### 4. Cargamos el corpus del portal

In [5]:
with open(
    PORTAL_CORPUS,
    "r",
    encoding="utf-8"
) as f:

    portal_data = json.load(f)

print(
    f"Documentos del portal: {len(portal_data)}"
)

Documentos del portal: 32


In [6]:
# Comprobamos
portal_data[0]

{'id': 'portal_postgrado_0001',
 'fuente': 'portal',
 'categoria': 'postgrado',
 'titulo': 'Estudios de Postgrado UPT Aragua',
 'url': 'https://upta.edu.ve/postgrado',
 'fecha_extraccion': '2026-08-12',
 'texto': '+58 (0243) 246-2211\n[email\xa0protected]\nSíguenos:\n|\nPortal Estudiantil\nIntranet Docente\nPostgrado\nOferta Académica de Postgrado\nEstudios de Postgrado\nUPT Aragua\nOfrecemos programas de postgrado o formación avanzada (PNFA) y diplomados orientados a la investigación, la innovación y el desarrollo tecnológico, en áreas estratégicas para la industria venezolana.\n4\nProgramas PNFA\n6\nDiplomados\n50+\nAños de trayectoria\nEspecializaciones · Maestrías · Doctorados\nPostgrados o Programas Nacionales de Formación Avanzada (PNFA)\nInscripción Abierta\nPostgrado en Informática\nMención Desarrollo de Software\nEspecialización\nMaestría\nDoctorado\nForma investigadores y profesionales de alto nivel en el desarrollo de software, inteligencia artificial y sistemas de informaci

### 5. Cargamos también los TXT de los documentos institucionales

Aquí vamos a integrar los PDFs que ya extrajimos anteriormente.

In [7]:
DOCUMENTOS_DIR = os.path.join(
    BASE_DIR,
    "textoLimpio"
)

In [8]:
def cargar_documentos_txt(base_dir):

    documentos = []

    for root, dirs, files in os.walk(base_dir):

        for file in files:

            if not file.lower().endswith(".txt"):
                continue

            path = os.path.join(
                root,
                file
            )

            try:

                with open(
                    path,
                    "r",
                    encoding="utf-8"
                ) as f:

                    texto = f.read()

                relativa = os.path.relpath(
                    root,
                    base_dir
                )

                partes = relativa.split(
                    os.sep
                )

                categoria = (
                    partes[0]
                    if partes and partes[0] != "."
                    else "institucional"
                )

                documentos.append({

                    "id": None,

                    "fuente": "documento",

                    "categoria": categoria,

                    "titulo": os.path.splitext(
                        file
                    )[0],

                    "url": None,

                    "documento": file,

                    "ruta": path,

                    "fecha_extraccion": None,

                    "texto": texto
                })

            except Exception as e:

                print(
                    f"Error leyendo {path}: {e}"
                )

    return documentos

In [12]:
# Ejecutamos
documentos_data = cargar_documentos_txt(
    DOCUMENTOS_DIR
)

print(
    f"Documentos institucionales: "
    f"{len(documentos_data)}"
)

Documentos institucionales: 10


### 6. Normalizar los registros del portal

Queremos que portal y documentos tengan exactamente la misma estructura.

In [13]:
def normalizar_registro_portal(item):

    return {

        "id": item.get("id"),

        "fuente": "portal",

        "categoria": item.get(
            "categoria",
            "general"
        ),

        "titulo": item.get(
            "titulo",
            ""
        ),

        "url": item.get(
            "url"
        ),

        "documento": None,

        "ruta": None,

        "fecha_extraccion": item.get(
            "fecha_extraccion"
        ),

        "texto": item.get(
            "texto",
            ""
        )
    }

In [10]:
portal_normalizado = [
    normalizar_registro_portal(x)
    for x in portal_data
]

In [14]:
# Combinamos
corpus = (
    portal_normalizado +
    documentos_data
)

print(
    f"Corpus total: {len(corpus)} documentos"
)

Corpus total: 42 documentos


### 7. Normalización profunda del texto

Ahora empieza la parte importante.

In [15]:
def normalizar_unicode(texto):

    texto = unicodedata.normalize(
        "NFKC",
        texto
    )

    return texto

In [16]:
# Eliminamos caracteres problemáticos
def limpiar_caracteres(texto):

    texto = texto.replace(
        "\x00",
        " "
    )

    texto = texto.replace(
        "\ufeff",
        ""
    )

    texto = texto.replace(
        "\u00a0",
        " "
    )

    return texto

In [17]:
# Normalizamos espacios
def normalizar_espacios(texto):

    texto = re.sub(
        r"[ \t]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\n[ \t]+",
        "\n",
        texto
    )

    texto = re.sub(
        r"\n{3,}",
        "\n\n",
        texto
    )

    return texto.strip()

### 8. Corregir palabras divididas por PDF

Este problema aparece muchísimo:

In [18]:
def unir_palabras_partidas(texto):

    patron = r"(\w)-\n(\w)"

    while re.search(
        patron,
        texto
    ):

        texto = re.sub(
            patron,
            r"\1\2",
            texto
        )

    return texto

### 9. Normalización completa

In [19]:
def normalizar_texto(texto):

    if not texto:
        return ""

    texto = normalizar_unicode(
        texto
    )

    texto = limpiar_caracteres(
        texto
    )

    texto = unir_palabras_partidas(
        texto
    )

    texto = normalizar_espacios(
        texto
    )

    return texto

In [20]:
# Aplicamos
for item in tqdm(corpus):

    item["texto"] = normalizar_texto(
        item["texto"]
    )

    item["titulo"] = normalizar_texto(
        item["titulo"]
    )

  0%|          | 0/42 [00:00<?, ?it/s]

10. Detectar encabezados y títulos

Ahora intentaremos identificar líneas que probablemente sean títulos.

Ejemplos:
PROGRAMAS NACIONALES DE FORMACIÓN AVANZADA

Requisitos de ingreso

Perfil del egresado

Objetivos

PLAN DE ESTUDIOS

In [21]:
def parece_titulo(linea):

    linea = linea.strip()

    if not linea:
        return False

    if len(linea) > 150:
        return False

    palabras = linea.split()

    if len(palabras) > 20:
        return False

    # Títulos en mayúsculas
    letras = [
        c for c in linea
        if c.isalpha()
    ]

    if letras:

        mayusculas = [
            c for c in letras
            if c.isupper()
        ]

        porcentaje = (
            len(mayusculas)
            / len(letras)
        )

        if porcentaje > 0.75:
            return True

    # Patrones habituales
    patrones = [

        r"^objetivos?$",

        r"^objetivo general$",

        r"^objetivos específicos$",

        r"^requisitos?$",

        r"^perfil$",

        r"^perfil de ingreso$",

        r"^perfil del egresado$",

        r"^plan de estudios$",

        r"^pensum$",

        r"^malla curricular$",

        r"^admisión$",

        r"^inscripción$",

        r"^contacto$",

        r"^información de contacto$",

        r"^duración$",

        r"^modalidad$",

        r"^servicios?$",

        r"^procedimiento$",

        r"^procedimientos?$"
    ]

    for patron in patrones:

        if re.match(
            patron,
            linea,
            re.IGNORECASE
        ):

            return True

    return False

11. Detectar listas

Esto es particularmente importante para requisitos.

Ejemplo:
- Copia de la cédula
- Título universitario
- Fondo negro del título

También:

1. Completar formulario
2. Entregar documentos
3. Esperar validación

In [22]:
def es_elemento_lista(linea):

    linea = linea.strip()

    patrones = [

        r"^[-•▪●]\s+",

        r"^\d+[\.\)]\s+",

        r"^[a-zA-Z][\.\)]\s+",

        r"^[ivxIVX]+[\.\)]\s+"
    ]

    return any(
        re.match(
            patron,
            linea
        )
        for patron in patrones
    )

### 12. Detectar pasos de procedimiento

In [23]:
def parece_paso(linea):

    patrones = [

        r"^(paso\s+)?\d+[\.\:\-]",

        r"^primero",

        r"^segundo",

        r"^tercero",

        r"^cuarto",

        r"^quinto",

        r"^posteriormente",

        r"^finalmente",

        r"^a continuación"
    ]

    return any(
        re.match(
            patron,
            linea,
            re.IGNORECASE
        )
        for patron in patrones
    )

### 13. Detectar teléfonos

Esto será muy útil para el chatbot.

In [25]:
PATRON_TELEFONO = re.compile(
    r"""
    (?: # Optional country code
        \+?\d{1,4}
        [\s\-]?
    )?
    (?: # Optional area code
        \(?\d{3,4}\)?
        [\s\-]?
    )
    \d{3,4}
    [\s\-]?
    \d{3,4}
    """,
    re.VERBOSE
)

In [26]:
def extraer_telefonos(texto):

    return PATRON_TELEFONO.findall(
        texto
    )

### 14. Detectar correos

In [27]:
PATRON_EMAIL = re.compile(
    r"\b[A-Za-z0-9._%+-]+"
    r"@[A-Za-z0-9.-]+\."
    r"[A-Za-z]{2,}\b"
)

In [28]:
def extraer_emails(texto):

    return PATRON_EMAIL.findall(
        texto
    )

### 15. Detectar URLs

In [29]:
PATRON_URL = re.compile(
    r"https?://[^\s]+"
)

In [30]:
def extraer_urls(texto):

    return PATRON_URL.findall(
        texto
    )

### 16. Detectar fechas

Para el español:

In [31]:
MESES = (
    "enero|febrero|marzo|abril|mayo|"
    "junio|julio|agosto|septiembre|"
    "octubre|noviembre|diciembre"
)

PATRON_FECHA = re.compile(
    rf"""
    (?:
        \d{{1,2}}
        \s+de\s+
        ({MESES})
        \s+de\s+
        \d{{4}}
    )
    |
    (?:
        \d{{1,2}}/
        \d{{1,2}}/
        \d{{2,4}}
    )
    |
    (?:
        \d{{1,2}}-
        \d{{1,2}}-
        \d{{2,4}}
    )
    """,
    re.IGNORECASE |
    re.VERBOSE
)

In [32]:
def extraer_fechas(texto):

    return [
        x.group(0)
        for x in PATRON_FECHA.finditer(
            texto
        )
    ]

### 17. Detectar montos

Para información de aranceles:

In [33]:
PATRON_MONTO = re.compile(
    r"""
    (?:
        (?:Bs\.?|VES|USD|\$)
        \s*
    )?
    \d{1,3}
    (?:
        [\.,]\d{3}
    )*
    (?:
        [\.,]\d{1,2}
    )?
    \s*
    (?:
        Bs\.?|VES|USD|\$|dólares?
    )?
    """,
    re.IGNORECASE |
    re.VERBOSE
)

In [34]:
def extraer_montos(texto):

    encontrados = []

    for match in PATRON_MONTO.finditer(
        texto
    ):

        valor = match.group(0).strip()

        if re.search(
            r"\d",
            valor
        ):

            encontrados.append(
                valor
            )

    return encontrados

### 18. Detectar entidades importantes

Podemos clasificar automáticamente algunos fragmentos

In [35]:
def detectar_tipo_fragmento(texto):

    texto_lower = texto.lower()

    if (
        "requisito" in texto_lower
        or "requisitos" in texto_lower
    ):

        return "requisitos"

    if (
        "inscripción" in texto_lower
        or "inscripcion" in texto_lower
        or "registro" in texto_lower
    ):

        return "inscripcion"

    if (
        "objetivo" in texto_lower
        or "objetivos" in texto_lower
    ):

        return "objetivos"

    if (
        "perfil de ingreso" in texto_lower
    ):

        return "perfil_ingreso"

    if (
        "perfil del egresado" in texto_lower
        or "perfil de egreso" in texto_lower
    ):

        return "perfil_egreso"

    if (
        "contacto" in texto_lower
        or PATRON_EMAIL.search(texto)
        or PATRON_TELEFONO.search(texto)
    ):

        return "contacto"

    if (
        "calendario" in texto_lower
        or "fecha" in texto_lower
    ):

        return "fechas"

    if (
        "arancel" in texto_lower
        or "precio" in texto_lower
        or "costo" in texto_lower
    ):

        return "aranceles"

    if (
        "procedimiento" in texto_lower
        or parece_paso(texto)
    ):

        return "procedimiento"

    if es_elemento_lista(texto):

        return "lista"

    return "general"

19. El corazón del pipeline: segmentación semántica

No queremos:
cada 2500 caracteres
Se quiere:
Título
   ↓
contenido relacionado

Subtítulo
   ↓
contenido relacionado

Requisitos
   ↓
todos los requisitos

Procedimiento
   ↓
pasos relacionados

In [36]:
def segmentar_semanticamente(texto):

    lineas = [
        x.strip()
        for x in texto.splitlines()
        if x.strip()
    ]

    fragmentos = []

    titulo_actual = None

    buffer = []

    def guardar_buffer():

        nonlocal buffer

        if not buffer:
            return

        contenido = "\n".join(
            buffer
        ).strip()

        if contenido:

            fragmentos.append({

                "seccion":
                    titulo_actual,

                "texto":
                    contenido
            })

        buffer = []

    for linea in lineas:

        if parece_titulo(linea):

            guardar_buffer()

            titulo_actual = linea

        else:

            buffer.append(linea)

    guardar_buffer()

    return fragmentos

### 20. Problema: fragmentos demasiado pequeños

Por ejemplo:
Requisitos
Título universitario.

Eso está bien.
Pero podemos tener:
Modalidad Presencial.

También está bien.
Sin embargo, a veces aparecen fragmentos de una sola palabra.
Vamos a filtrarlos.

In [37]:
def filtrar_fragmentos(fragmentos):

    resultado = []

    for fragmento in fragmentos:

        texto = fragmento["texto"].strip()

        palabras = texto.split()

        if len(palabras) < 5:

            continue

        resultado.append(
            fragmento
        )

    return resultado

### 21. Evitar fragmentos gigantes

Ahora tenemos el problema contrario.

Por ejemplo:
un capítulo completo de 15.000 palabras

No queremos eso.
Pero tampoco queremos cortar arbitrariamente.
Usaremos una segunda segmentación respetando párrafos/listas.

In [38]:
def dividir_fragmento_grande(
    texto,
    max_palabras=450
):

    palabras = texto.split()

    if len(palabras) <= max_palabras:

        return [texto]

    bloques = []

    actual = []

    contador = 0

    for palabra in palabras:

        actual.append(
            palabra
        )

        contador += 1

        if contador >= max_palabras:

            bloques.append(
                " ".join(actual)
            )

            actual = []

            contador = 0

    if actual:

        bloques.append(
            " ".join(actual)
        )

    return bloques

### Esto es una segunda barrera, no la segmentación principal

### 22. Construir fragmentos finales

In [39]:
def construir_fragmentos(
    registro
):

    texto = registro["texto"]

    segmentos = segmentar_semanticamente(
        texto
    )

    segmentos = filtrar_fragmentos(
        segmentos
    )

    fragmentos_finales = []

    indice = 1

    for segmento in segmentos:

        partes = dividir_fragmento_grande(
            segmento["texto"]
        )

        for parte in partes:

            metadata = {

                "tipo": detectar_tipo_fragmento(
                    parte
                ),

                "emails":
                    extraer_emails(
                        parte
                    ),

                "telefonos":
                    extraer_telefonos(
                        parte
                    ),

                "urls":
                    extraer_urls(
                        parte
                    ),

                "fechas":
                    extraer_fechas(
                        parte
                    ),

                "montos":
                    extraer_montos(
                        parte
                    )
            }

            fragmento = {

                "fragmento_id":
                    f"{registro['id']}_"
                    f"{indice:04d}",

                "fuente":
                    registro["fuente"],

                "categoria":
                    registro["categoria"],

                "titulo_documento":
                    registro["titulo"],

                "documento":
                    registro["documento"],

                "url":
                    registro["url"],

                "fecha_extraccion":
                    registro[
                        "fecha_extraccion"
                    ],

                "seccion":
                    segmento["seccion"],

                "tipo":
                    metadata["tipo"],

                "texto":
                    parte,

                "entidades": metadata
            }

            fragmentos_finales.append(
                fragmento
            )

            indice += 1

    return fragmentos_finales

### 23. Ejecutar sobre todo el corpus

In [40]:
fragmentos = []

for registro in tqdm(
    corpus,
    desc="Procesando corpus"
):

    try:

        resultado = construir_fragmentos(
            registro
        )

        fragmentos.extend(
            resultado
        )

    except Exception as e:

        print(
            f"Error en "
            f"{registro.get('id')}: "
            f"{e}"
        )

print(
    f"Fragmentos generados: "
    f"{len(fragmentos)}"
)

Procesando corpus:   0%|          | 0/42 [00:00<?, ?it/s]

Fragmentos generados: 1043


### 24. Guardar JSONL

Para procesamiento posterior es preferible JSONL.

In [41]:
jsonl_path = os.path.join(
    SEMANTIC_DIR,
    "fragmentos_semanticos.jsonl"
)

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as f:

    for fragmento in fragmentos:

        f.write(
            json.dumps(
                fragmento,
                ensure_ascii=False
            )
            + "\n"
        )

print(
    "Guardado:",
    jsonl_path
)

Guardado: /content/drive/MyDrive/UniversidadLLM/semantico/fragmentos_semanticos.jsonl


### 25. Guardar JSON normal

In [42]:
json_path = os.path.join(
    SEMANTIC_DIR,
    "fragmentos_semanticos.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fragmentos,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Guardado:",
    json_path
)

Guardado: /content/drive/MyDrive/UniversidadLLM/semantico/fragmentos_semanticos.json


### 26. Crear estadísticas

Ahora podemos saber qué tenemos realmente

In [43]:
df = pd.DataFrame(
    fragmentos
)

df["palabras"] = (
    df["texto"]
    .str.split()
    .str.len()
)

df["caracteres"] = (
    df["texto"]
    .str.len()
)

**Por categorías**

In [44]:
df.groupby(
    "categoria"
).size().sort_values(
    ascending=False
)

,0
categoria,
programasPregrado,871
reglamentos,57
postgrado,54
investigacion,23
vinculacion,19
calendarioAcademico,13
admisiones,6


**Por tipo**

In [45]:
df.groupby(
    "tipo"
).size().sort_values(
    ascending=False
)

,0
tipo,
general,688
objetivos,79
inscripcion,58
requisitos,51
procedimiento,48
aranceles,37
contacto,28
lista,28
fechas,17


### 27. Estadísticas de entidades

In [46]:
print(
    "Fragmentos con email:",
    sum(
        bool(x["entidades"]["emails"])
        for x in fragmentos
    )
)

print(
    "Fragmentos con teléfono:",
    sum(
        bool(x["entidades"]["telefonos"])
        for x in fragmentos
    )
)

print(
    "Fragmentos con fechas:",
    sum(
        bool(x["entidades"]["fechas"])
        for x in fragmentos
    )
)

print(
    "Fragmentos con montos:",
    sum(
        bool(x["entidades"]["montos"])
        for x in fragmentos
    )
)

Fragmentos con email: 0
Fragmentos con teléfono: 56
Fragmentos con fechas: 44
Fragmentos con montos: 639


Esto es interesante porque después podremos generar los datasets especializados:
QA_CONTACTOS
QA_REQUISITOS
QA_PROCEDIMIENTOS
QA_PROGRAMAS
QA_FECHAS
QA_ARANCELES
QA_GENERAL

### 28. Crear archivo para revisión humana

Antes de entrenar absolutamente nada, yo revisaría manualmente una muestra.

In [47]:
revision = df[
    [
        "fragmento_id",
        "fuente",
        "categoria",
        "titulo_documento",
        "seccion",
        "tipo",
        "texto",
        "url"
    ]
].copy()

**Tomamos una muestra:**

In [48]:
muestra = revision.sample(
    min(100, len(revision)),
    random_state=42
)

**Guardamos**

In [49]:
revision_path = os.path.join(
    SEMANTIC_DIR,
    "revision_manual.csv"
)

muestra.to_csv(
    revision_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Archivo:",
    revision_path
)

Archivo: /content/drive/MyDrive/UniversidadLLM/semantico/revision_manual.csv


Revisar:

¿El fragmento tiene sentido?
¿El título está bien identificado?
¿La categoría es correcta?
¿Se conservaron las listas?
¿Se destruyeron tablas?
¿Se cortó información importante?

### 29. Una mejora que considero MUY importante para la universidad

Hay una cuestión que el código anterior todavía no resuelve completamente:

Las tablas.

Por ejemplo, imagina un PDF con:

Programa	Duración	Créditos
Informática	3 años	42
Mecánica	3 años	42

La extracción convencional puede producir algo parecido a:
Programa
Duración
Créditos
Informática
3 años
42
Mecánica
3 años
42

Eso no es suficiente para entrenar un buen modelo.

Debemos detectar las tablas y convertirlas en texto semántico:
Programa: Informática.
Duración: 3 años.
Créditos: 42.

Programa: Mecánica.
Duración: 3 años.
Créditos: 42.

O incluso:

El programa de Informática tiene una duración de 3 años
y contempla 42 créditos.

Esto mejora muchísimo la generación posterior de QA.




### 30. También se quieren preservar listas

Por ejemplo:
Requisitos de inscripción:

1. Copia de la cédula.
2. Título universitario.
3. Notas certificadas.
4. Fotografía.

Nuestro fragmento debe quedar como:
Sección: Requisitos de inscripción

1. Copia de la cédula.
2. Título universitario.
3. Notas certificadas.
4. Fotografía.

No debemos convertirlo todavía en preguntas.
Primero preservamos la información.

### 31. Resultado que buscamos

Al terminar esta etapa tendrás algo parecido a:
{
  "fragmento_id": "portal_postgrado_0007_0003",
  "fuente": "portal",
  "categoria": "postgrado",
  "titulo_documento": "Estudios de Postgrado",
  "url": "https://upta.edu.ve/postgrado",
  "seccion": "REQUISITOS",
  "tipo": "requisitos",
  "texto": "Para ingresar al programa se requiere...",
  "entidades": {
    "emails": [],
    "telefonos": [],
    "urls": [],
    "fechas": [],
    "montos": []
  }
}

Y otro podría ser:
{
  "fragmento_id": "portal_admisiones_0002_0005",
  "fuente": "portal",
  "categoria": "admisiones",
  "titulo_documento": "Inscripción",
  "url": "https://upta.edu.ve/admisiones/inscripcion",
  "seccion": "PROCEDIMIENTO",
  "tipo": "inscripcion",
  "texto": "1. Completar el formulario de inscripción...",
  "entidades": {
    "emails": ["..."],
    "telefonos": ["..."],
    "urls": [],
    "fechas": [],
    "montos": []
  }
}

**Eso ya es material de alta calidad para la siguiente etapa.**

La arquitectura que te recomiendo finalmente

Para el proyecto que estás construyendo, yo dejaría el pipeline así:

                         ┌──────────────────┐
                         │ PDFs institucionales │
                         └────────┬─────────┘
                                  │
                                  ▼
                         Extracción PDF
                                  │
                                  ▼
                         Texto normalizado
                                  │
                                  │
┌─────────────────┐               │
│  upta.edu.ve    │               │
│                 │               │
│ Postgrado       │               │
│ Diplomados      │               │
│ Admisiones      │               │
│ Vinculación     │               │
│ Investigación   │               │
└────────┬────────┘               │
         │                        │
         ▼                        ▼
      Scraping              Corpus documental
         │                        │
         └──────────┬─────────────┘
                    ▼
          ┌─────────────────────┐
          │ NORMALIZACIÓN       │
          │ SEMÁNTICA           │
          └──────────┬──────────┘
                     │
                     ▼
          ┌─────────────────────┐
          │ Fragmentos          │
          │ estructurados       │
          └──────────┬──────────┘
                     │
       ┌─────────────┼─────────────┐
       ▼             ▼             ▼
  Requisitos     Procedimientos  Contactos
       │             │             │
       ▼             ▼             ▼
    Programas      Fechas       Aranceles
       │             │             │
       └─────────────┼─────────────┘
                     ▼
              CORPUS SEMÁNTICO
                     │
                     ▼
          GENERACIÓN DE QA
                     │
                     ▼
             CONTROL DE CALIDAD
                     │
                     ▼
              DATASET FINAL
                     │
                     ▼
                  QLoRA
                     │
                     ▼
             MODELO UPT ARAGUA

Y una recomendación clave

No pases todavía a la generación automática de 5.000–10.000 QA. Primero ejecuta este pipeline y revisa esos revision_manual.csv.

En particular, quiero que comprobemos cómo están saliendo los PDFs que contienen tablas y planes de estudio, porque ese será probablemente el punto que más tendremos que ajustar antes de generar el dataset. Una vez validemos esa capa, podemos hacer la siguiente etapa: un generador de QA especializado por tipo de fragmento, de manera que un fragmento de requisitos genere preguntas diferentes a uno de contacto, uno de programa académico o uno de procedimiento. Eso será mucho más potente que pedirle a un LLM que convierta todo el corpus indiscriminadamente en preguntas.